In [56]:
import numpy as np

In [57]:
class DecisionStump:
  def __init__(self):
    self.polarity=1
    self.feature_idx=None
    self.threshold=None
    self.alpha=None

  def predict(self, X):
    n_samples=X.shape[0]
    X_column=X[:, self.feature_idx]

    predictions=np.ones(n_samples)
    if (self.polarity==1):
      predictions[X_column<self.threshold]=-1
    else:
      predictions[X_column>=self.threshold]=-1

    return predictions

class AdaBoost:
  def __init__(self, n_clf=50):
    self.n_clf=n_clf

  def fit(self, X, y):
    n_samples,n_features=X.shape

    w=np.full(n_samples, (1/n_samples))
    self.clfs=[]

    for _ in range(self.n_clf):
      clf=DecisionStump()
      min_error=float("inf")

      for feature_i in range(n_features):
        X_column=X[:, feature_i]
        thresholds=np.unique(X_column)

        for threshold in thresholds:
          p=1
          predictions=np.ones(n_samples)
          predictions[X_column<threshold]=-1

          missclassified = w[y!=predictions]
          error=sum(missclassified)

          if (error > 0.5):
            error=1-error
            p=-1

          if (error<min_error):
            min_error=error
            clf.polarity=p
            clf.threshold=threshold
            clf.feature_idx=feature_i

      clf.alpha=0.5*np.log((1.0-min_error)/(min_error+1e-10))
      predictions=clf.predict(X)
      w*=np.exp(-clf.alpha*y*predictions)
      w/=np.sum(w)

      self.clfs.append(clf)

  def predict(self, X):
    clf_preds=[clf.alpha*clf.predict(X) for clf in self.clfs]
    y_pred=np.sum(clf_preds, axis=0)
    y_pred=np.sign(y_pred)
    return y_pred

In [52]:
!pip install kaggle
!kaggle datasets download -d uciml/mushroom-classification --unzip

Dataset URL: https://www.kaggle.com/datasets/uciml/mushroom-classification
License(s): CC0-1.0
100% 34.2k/34.2k [00:00<00:00, 51.6MB/s]



In [53]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, precision_score, confusion_matrix
from sklearn.linear_model import LogisticRegression

In [54]:
df = pd.read_csv("mushrooms.csv")
df = df.drop_duplicates()
y = df["class"].map({"e": -1, "p": 1}).to_numpy()
X = df.drop(columns=["class"])
X = pd.get_dummies(X, drop_first=True)
X = X.astype(float).to_numpy()
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (8124, 95)
y shape: (8124,)


In [55]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train

array([[0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.]], shape=(6499, 95))

In [58]:
model = AdaBoost(n_clf=50)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print(
    "\nClassification Report:\n",
    classification_report(y_test, y_pred, target_names=["Edible (-1)", "Poisonous (1)"]),
)

Test Accuracy: 0.9963

Confusion Matrix:
 [[842   0]
 [  6 777]]

Classification Report:
                precision    recall  f1-score   support

  Edible (-1)       0.99      1.00      1.00       842
Poisonous (1)       1.00      0.99      1.00       783

     accuracy                           1.00      1625
    macro avg       1.00      1.00      1.00      1625
 weighted avg       1.00      1.00      1.00      1625

